In [24]:
import pandas as pd
import gurobipy as gp
from gurobipy import GRB
from IPython.display import display, HTML

In [25]:
# -----------------------------
# 1-1. SKU 데이터
# -----------------------------

skus_df = pd.DataFrame([
    {"sku": "P1", "item": "item A", "weight_per_unit": 1.0},
    {"sku": "P2", "item": "item B", "weight_per_unit": 0.8},
    {"sku": "P3", "item": "item C", "weight_per_unit": 0.5},
    {"sku": "P4", "item": "item D", "weight_per_unit": 1.2},
])

display(skus_df)

,sku,item,weight_per_unit
0,P1,item A,1.0
1,P2,item B,0.8
2,P3,item C,0.5
3,P4,item D,1.2


In [26]:
# -----------------------------
# 1-2. 수하주 데이터
# -----------------------------

buyers_df = pd.DataFrame([
    {"buyer_id": "B1", "buyer_name": "수하주1", "buyer_region": "SEOUL"},
    {"buyer_id": "B2", "buyer_name": "수하주2", "buyer_region": "GYEONGGI"},
    {"buyer_id": "B3", "buyer_name": "수하주3", "buyer_region": "GANGWON"},
    {"buyer_id": "B4", "buyer_name": "수하주4", "buyer_region": "SEOUL"},
])

display(buyers_df)

,buyer_id,buyer_name,buyer_region
0,B1,수하주1,SEOUL
1,B2,수하주2,GYEONGGI
2,B3,수하주3,GANGWON
3,B4,수하주4,SEOUL


In [27]:
# -----------------------------
# 1-3. 주문 데이터
# -----------------------------

orders_df = pd.DataFrame([
    {
        "order_id": "O1",
        "buyer_id": "B1",
        "time_window": "T1_AM",
        "budget": 500000,
        "split_allowed": True,
    },
    {
        "order_id": "O2",
        "buyer_id": "B2",
        "time_window": "T1_AM",
        "budget": 400000,
        "split_allowed": True,
    },
    {
        "order_id": "O3",
        "buyer_id": "B1",
        "time_window": "T2_PM",
        "budget": 350000,
        "split_allowed": True,
    },
    {
        "order_id": "O4",
        "buyer_id": "B3",
        "time_window": "T2_PM",
        "budget": 600000,
        "split_allowed": True,
    },
    {
        "order_id": "O5",
        "buyer_id": "B4",
        "time_window": "T1_AM",
        "budget": 450000,
        "split_allowed": True,
    },
])

display(orders_df)

,order_id,buyer_id,time_window,budget,split_allowed
0,O1,B1,T1_AM,500000,True
1,O2,B2,T1_AM,400000,True
2,O3,B1,T2_PM,350000,True
3,O4,B3,T2_PM,600000,True
4,O5,B4,T1_AM,450000,True


In [28]:
# -----------------------------
# 1-4. 주문별 SKU 수요 데이터
# -----------------------------

order_items_df = pd.DataFrame([
    {"order_id": "O1", "sku": "P1", "demand": 50, "unit_bid": 6000, "min_fill_rate": 0.8},
    {"order_id": "O1", "sku": "P2", "demand": 30, "unit_bid": 7000, "min_fill_rate": 0.8},

    {"order_id": "O2", "sku": "P1", "demand": 40, "unit_bid": 5800, "min_fill_rate": 0.8},
    {"order_id": "O2", "sku": "P3", "demand": 60, "unit_bid": 4200, "min_fill_rate": 0.8},

    {"order_id": "O3", "sku": "P2", "demand": 40, "unit_bid": 6800, "min_fill_rate": 0.8},
    {"order_id": "O3", "sku": "P4", "demand": 20, "unit_bid": 8000, "min_fill_rate": 0.8},

    {"order_id": "O4", "sku": "P1", "demand": 70, "unit_bid": 6200, "min_fill_rate": 0.8},
    {"order_id": "O4", "sku": "P3", "demand": 80, "unit_bid": 4500, "min_fill_rate": 0.8},

    {"order_id": "O5", "sku": "P2", "demand": 35, "unit_bid": 7200, "min_fill_rate": 0.8},
    {"order_id": "O5", "sku": "P4", "demand": 25, "unit_bid": 8500, "min_fill_rate": 0.8},
])

display(order_items_df)

,order_id,sku,demand,unit_bid,min_fill_rate
0,O1,P1,50,6000,0.8
1,O1,P2,30,7000,0.8
2,O2,P1,40,5800,0.8
3,O2,P3,60,4200,0.8
4,O3,P2,40,6800,0.8
5,O3,P4,20,8000,0.8
6,O4,P1,70,6200,0.8
7,O4,P3,80,4500,0.8
8,O5,P2,35,7200,0.8
9,O5,P4,25,8500,0.8


In [29]:
# -----------------------------
# 1-5. 화주 데이터
# -----------------------------

sellers_df = pd.DataFrame([
    {"seller_id": "S1", "seller_name": "화주1", "seller_region": "SEOUL"},
    {"seller_id": "S2", "seller_name": "화주2", "seller_region": "GYEONGGI"},
    {"seller_id": "S3", "seller_name": "화주3", "seller_region": "GANGWON"},
    {"seller_id": "S4", "seller_name": "화주4", "seller_region": "GYEONGGI"},
])

display(sellers_df)

,seller_id,seller_name,seller_region
0,S1,화주1,SEOUL
1,S2,화주2,GYEONGGI
2,S3,화주3,GANGWON
3,S4,화주4,GYEONGGI


In [30]:
# -----------------------------
# 1-6. 화주별 SKU 재고 데이터
# -----------------------------

seller_inventory_df = pd.DataFrame([
    {"seller_id": "S1", "sku": "P1", "inventory": 100, "ask_price": 4000, "lot_min_ratio": 0.2},
    {"seller_id": "S1", "sku": "P2", "inventory": 50,  "ask_price": 5200, "lot_min_ratio": 0.2},

    {"seller_id": "S2", "sku": "P1", "inventory": 80,  "ask_price": 4300, "lot_min_ratio": 0.2},
    {"seller_id": "S2", "sku": "P3", "inventory": 120, "ask_price": 3000, "lot_min_ratio": 0.2},

    {"seller_id": "S3", "sku": "P1", "inventory": 90,  "ask_price": 3800, "lot_min_ratio": 0.2},
    {"seller_id": "S3", "sku": "P4", "inventory": 70,  "ask_price": 6000, "lot_min_ratio": 0.2},

    {"seller_id": "S4", "sku": "P2", "inventory": 100, "ask_price": 5000, "lot_min_ratio": 0.2},
    {"seller_id": "S4", "sku": "P4", "inventory": 60,  "ask_price": 6200, "lot_min_ratio": 0.2},
])

display(seller_inventory_df)

,seller_id,sku,inventory,ask_price,lot_min_ratio
0,S1,P1,100,4000,0.2
1,S1,P2,50,5200,0.2
2,S2,P1,80,4300,0.2
3,S2,P3,120,3000,0.2
4,S3,P1,90,3800,0.2
5,S3,P4,70,6000,0.2
6,S4,P2,100,5000,0.2
7,S4,P4,60,6200,0.2


In [31]:
# -----------------------------
# 2. 권역별 예상 배송비 데이터
# -----------------------------

region_cost_df = pd.DataFrame([
    # seller_region, buyer_region, fixed_cost, variable_cost_per_weight

    {"seller_region": "SEOUL",    "buyer_region": "SEOUL",    "fixed_cost": 3000,  "var_cost_per_weight": 30},
    {"seller_region": "SEOUL",    "buyer_region": "GYEONGGI", "fixed_cost": 5000,  "var_cost_per_weight": 50},
    {"seller_region": "SEOUL",    "buyer_region": "GANGWON",  "fixed_cost": 9000,  "var_cost_per_weight": 90},

    {"seller_region": "GYEONGGI", "buyer_region": "SEOUL",    "fixed_cost": 5000,  "var_cost_per_weight": 50},
    {"seller_region": "GYEONGGI", "buyer_region": "GYEONGGI", "fixed_cost": 3500,  "var_cost_per_weight": 35},
    {"seller_region": "GYEONGGI", "buyer_region": "GANGWON",  "fixed_cost": 8500,  "var_cost_per_weight": 85},

    {"seller_region": "GANGWON",  "buyer_region": "SEOUL",    "fixed_cost": 9000,  "var_cost_per_weight": 90},
    {"seller_region": "GANGWON",  "buyer_region": "GYEONGGI", "fixed_cost": 8500,  "var_cost_per_weight": 85},
    {"seller_region": "GANGWON",  "buyer_region": "GANGWON",  "fixed_cost": 5000,  "var_cost_per_weight": 50},
])

display(region_cost_df)

,seller_region,buyer_region,fixed_cost,var_cost_per_weight
0,SEOUL,SEOUL,3000,30
1,SEOUL,GYEONGGI,5000,50
2,SEOUL,GANGWON,9000,90
3,GYEONGGI,SEOUL,5000,50
4,GYEONGGI,GYEONGGI,3500,35
5,GYEONGGI,GANGWON,8500,85
6,GANGWON,SEOUL,9000,90
7,GANGWON,GYEONGGI,8500,85
8,GANGWON,GANGWON,5000,50


In [32]:
# -----------------------------
# 3. surplus 배분 비율
# -----------------------------

platform_surplus_share = 0.2
seller_surplus_share = 0.4
buyer_surplus_share = 0.4

assert abs(
    platform_surplus_share
    + seller_surplus_share
    + buyer_surplus_share
    - 1.0
) < 1e-9

In [33]:
# -----------------------------
# 4. 시간창 주문 batch 생성
# -----------------------------

def build_order_batch(
    target_time_window,
    orders_df,
    order_items_df,
    buyers_df,
    skus_df
):
    """
    target_time_window에 해당하는 주문과 주문 SKU 데이터를 결합한다.
    """

    orders_tw = orders_df[
        orders_df["time_window"] == target_time_window
    ].copy()

    if orders_tw.empty:
        return pd.DataFrame()

    batch_df = (
        order_items_df
        .merge(orders_tw, on="order_id", how="inner")
        .merge(buyers_df, on="buyer_id", how="left")
        .merge(skus_df, on="sku", how="left")
    )

    return batch_df

In [34]:
# -----------------------------
# 5. 가능한 거래 조합 생성
# -----------------------------

def build_valid_trade_tuples(
    order_batch_df,
    seller_inventory_df,
    sellers_df,
    region_cost_df
):
    """
    가능한 거래 조합 A_t를 생성한다.
    각 row가 (order_id, seller_id, sku)에 해당한다.
    """

    if order_batch_df.empty:
        return pd.DataFrame()

    seller_supply_df = (
        seller_inventory_df
        .merge(sellers_df, on="seller_id", how="left")
    )

    # SKU 기준으로 주문 수요와 화주 공급 후보 결합
    candidates = (
        order_batch_df
        .merge(seller_supply_df, on="sku", how="inner")
    )

    # 예상 배송비 결합
    candidates = candidates.merge(
        region_cost_df,
        on=["seller_region", "buyer_region"],
        how="left"
    )

    # feasible 조건
    candidates = candidates[
        (candidates["demand"] > 0) &
        (candidates["inventory"] > 0) &
        (candidates["unit_bid"] >= candidates["ask_price"])
    ].copy()

    candidates["M"] = candidates[["demand", "inventory"]].min(axis=1)

    candidates["unit_trade_surplus"] = (
        candidates["unit_bid"] - candidates["ask_price"]
    )

    candidates["unit_weight"] = candidates["weight_per_unit"]

    return candidates.reset_index(drop=True)

In [35]:
# -----------------------------
# 6. 1단계 WDP-based 거래 매칭 모델
# -----------------------------

def solve_stage1_wdp_trading_matching_df(
    target_time_window,
    orders_df,
    order_items_df,
    buyers_df,
    sellers_df,
    seller_inventory_df,
    skus_df,
    region_cost_df,
    eta=1.0,
    margin_mu=0.0,
    write_lp=True
):
    """
    1단계 WDP-based trading matching model.
    table 기반 데이터 구조 버전.
    """

    # -----------------------------
    # 6-1. 데이터 생성
    # -----------------------------
    order_batch_df = build_order_batch(
        target_time_window,
        orders_df,
        order_items_df,
        buyers_df,
        skus_df
    )

    if order_batch_df.empty:
        print(f"[{target_time_window}] 해당 시간창에 주문이 없습니다.")
        return None

    A_df = build_valid_trade_tuples(
        order_batch_df,
        seller_inventory_df,
        sellers_df,
        region_cost_df
    )

    if A_df.empty:
        print(f"[{target_time_window}] 가능한 거래 조합이 없습니다.")
        return None

    # -----------------------------
    # 6-2. 집합 생성
    # -----------------------------
    O_t = sorted(order_batch_df["order_id"].unique().tolist())

    P_o = {
        o: sorted(order_batch_df.loc[
            order_batch_df["order_id"] == o, "sku"
        ].unique().tolist())
        for o in O_t
    }

    A = [
        (row.order_id, row.seller_id, row.sku)
        for row in A_df.itertuples(index=False)
    ]

    seller_sku_pairs = sorted(
        set((row.seller_id, row.sku) for row in A_df.itertuples(index=False))
    )

    Gamma = sorted(
        set((row.buyer_id, row.seller_id) for row in A_df.itertuples(index=False))
    )

    # -----------------------------
    # 6-3. 파라미터 dict 생성
    # -----------------------------
    demand = {
        (row.order_id, row.sku): row.demand
        for row in order_batch_df.itertuples(index=False)
    }

    unit_bid = {
        (row.order_id, row.sku): row.unit_bid
        for row in order_batch_df.itertuples(index=False)
    }

    min_fill_rate = {
        (row.order_id, row.sku): row.min_fill_rate
        for row in order_batch_df.itertuples(index=False)
    }

    budget = {
        row.order_id: row.budget
        for row in order_batch_df.drop_duplicates("order_id").itertuples(index=False)
    }

    split_allowed = {
        row.order_id: row.split_allowed
        for row in order_batch_df.drop_duplicates("order_id").itertuples(index=False)
    }

    buyer_of_order = {
        row.order_id: row.buyer_id
        for row in order_batch_df.drop_duplicates("order_id").itertuples(index=False)
    }

    ask_price = {
        (row.seller_id, row.sku): row.ask_price
        for row in A_df.drop_duplicates(["seller_id", "sku"]).itertuples(index=False)
    }

    inventory = {
        (row.seller_id, row.sku): row.inventory
        for row in A_df.drop_duplicates(["seller_id", "sku"]).itertuples(index=False)
    }

    lot_min_ratio = {
        (row.seller_id, row.sku): row.lot_min_ratio
        for row in A_df.drop_duplicates(["seller_id", "sku"]).itertuples(index=False)
    }

    M = {
        (row.order_id, row.seller_id, row.sku): row.M
        for row in A_df.itertuples(index=False)
    }

    fixed_cost = {
        (row.buyer_id, row.seller_id): row.fixed_cost
        for row in A_df.drop_duplicates(["buyer_id", "seller_id"]).itertuples(index=False)
    }

    var_cost_per_weight = {
        (row.buyer_id, row.seller_id): row.var_cost_per_weight
        for row in A_df.drop_duplicates(["buyer_id", "seller_id"]).itertuples(index=False)
    }

    weight_per_unit = {
        row.sku: row.weight_per_unit
        for row in skus_df.itertuples(index=False)
    }

    # -----------------------------
    # 6-4. 모델 생성
    # -----------------------------
    model = gp.Model(f"stage1_wdp_trading_matching_{target_time_window}")

    x = model.addVars(O_t, vtype=GRB.BINARY, name="x_order")
    q = model.addVars(A, vtype=GRB.CONTINUOUS, lb=0, name="q")
    z = model.addVars(A, vtype=GRB.BINARY, name="z")
    y = model.addVars(seller_sku_pairs, vtype=GRB.BINARY, name="y_seller_sku")
    w = model.addVars(Gamma, vtype=GRB.BINARY, name="w_buyer_seller")

    Q_index = [(o, p) for o in O_t for p in P_o[o]]
    Q = model.addVars(Q_index, vtype=GRB.CONTINUOUS, lb=0, name="Q")

    # -----------------------------
    # 6-5. 목적식
    # -----------------------------
    trade_surplus_expr = gp.quicksum(
        (unit_bid[o, p] - ask_price[s, p]) * q[o, s, p]
        for (o, s, p) in A
    )

    estimated_fixed_delivery_cost_expr = gp.quicksum(
        fixed_cost[b, s] * w[b, s]
        for (b, s) in Gamma
    )

    estimated_variable_delivery_cost_expr = gp.quicksum(
        var_cost_per_weight[buyer_of_order[o], s]
        * weight_per_unit[p]
        * q[o, s, p]
        for (o, s, p) in A
    )

    estimated_delivery_cost_expr = (
        estimated_fixed_delivery_cost_expr
        + estimated_variable_delivery_cost_expr
    )

    model.setObjective(
        trade_surplus_expr - eta * estimated_delivery_cost_expr,
        GRB.MAXIMIZE
    )

    # -----------------------------
    # 6-6. 제약식
    # -----------------------------

    # (1) 주문별 SKU 총 이행 수량
    for o in O_t:
        for p in P_o[o]:
            model.addConstr(
                Q[o, p] ==
                gp.quicksum(
                    q[o2, s, p2]
                    for (o2, s, p2) in A
                    if o2 == o and p2 == p
                ),
                name=f"total_fulfilled_{o}_{p}"
            )

    # (2) 최소 충족률
    for o in O_t:
        for p in P_o[o]:
            model.addConstr(
                Q[o, p] >= min_fill_rate[o, p] * demand[o, p] * x[o],
                name=f"min_fill_{o}_{p}"
            )

    # (3) 수요 초과 방지
    for o in O_t:
        for p in P_o[o]:
            model.addConstr(
                Q[o, p] <= demand[o, p] * x[o],
                name=f"max_demand_{o}_{p}"
            )

    # (4) 화주 재고
    for (s, p) in seller_sku_pairs:
        model.addConstr(
            gp.quicksum(
                q[o, s2, p2]
                for (o, s2, p2) in A
                if s2 == s and p2 == p
            )
            <= inventory[s, p],
            name=f"inventory_{s}_{p}"
        )

    # (5) q-z 연결
    for (o, s, p) in A:
        model.addConstr(
            q[o, s, p] <= M[o, s, p] * z[o, s, p],
            name=f"link_q_z_{o}_{s}_{p}"
        )

    # (6) z-x 연결
    for (o, s, p) in A:
        model.addConstr(
            z[o, s, p] <= x[o],
            name=f"link_z_x_{o}_{s}_{p}"
        )

    # (7) z-y 연결
    for (o, s, p) in A:
        model.addConstr(
            z[o, s, p] <= y[s, p],
            name=f"link_z_y_{o}_{s}_{p}"
        )

    # (8) lot 최소 거래 비율
    for (s, p) in seller_sku_pairs:
        model.addConstr(
            gp.quicksum(
                q[o, s2, p2]
                for (o, s2, p2) in A
                if s2 == s and p2 == p
            )
            >= lot_min_ratio[s, p] * inventory[s, p] * y[s, p],
            name=f"min_lot_ratio_{s}_{p}"
        )

    # (9) lot 사용 여부와 공급량 연결
    for (s, p) in seller_sku_pairs:
        model.addConstr(
            gp.quicksum(
                q[o, s2, p2]
                for (o, s2, p2) in A
                if s2 == s and p2 == p
            )
            <= inventory[s, p] * y[s, p],
            name=f"lot_usage_upper_{s}_{p}"
        )

    # (10) 주문별 예산
    for o in O_t:
        model.addConstr(
            gp.quicksum(
                unit_bid[o, p] * Q[o, p]
                for p in P_o[o]
            )
            <= budget[o] * x[o],
            name=f"budget_{o}"
        )

    # (11) 화주-수하주 연결 변수
    for (o, s, p) in A:
        b = buyer_of_order[o]
        model.addConstr(
            z[o, s, p] <= w[b, s],
            name=f"link_z_w_{o}_{s}_{p}"
        )

    for (b, s) in Gamma:
        model.addConstr(
            w[b, s] <= gp.quicksum(
                z[o, s2, p]
                for (o, s2, p) in A
                if s2 == s and buyer_of_order[o] == b
            ),
            name=f"link_w_z_{b}_{s}"
        )

    # (12) 예상 배송비 기반 적자 방지
    model.addConstr(
        trade_surplus_expr >= (1 + margin_mu) * estimated_delivery_cost_expr,
        name="non_deficit_estimated_delivery_cost"
    )

    # (13) split_allowed=False일 경우 같은 SKU 분할 공급 금지
    for o in O_t:
        if not split_allowed[o]:
            for p in P_o[o]:
                model.addConstr(
                    gp.quicksum(
                        z[o2, s, p2]
                        for (o2, s, p2) in A
                        if o2 == o and p2 == p
                    )
                    <= x[o],
                    name=f"no_split_{o}_{p}"
                )

    # -----------------------------
    # 6-7. LP 파일 출력 및 최적화
    # -----------------------------
    if write_lp:
        model.write(f"stage1_wdp_trading_matching_{target_time_window}.lp")

    model.optimize()

    return {
        "model": model,
        "target_time_window": target_time_window,
        "order_batch_df": order_batch_df,
        "A_df": A_df,
        "O_t": O_t,
        "P_o": P_o,
        "A": A,
        "Gamma": Gamma,
        "x": x,
        "q": q,
        "z": z,
        "y": y,
        "w": w,
        "Q": Q,
        "trade_surplus_expr": trade_surplus_expr,
        "estimated_delivery_cost_expr": estimated_delivery_cost_expr,
        "params": {
            "unit_bid": unit_bid,
            "ask_price": ask_price,
            "buyer_of_order": buyer_of_order,
            "fixed_cost": fixed_cost,
            "var_cost_per_weight": var_cost_per_weight,
            "weight_per_unit": weight_per_unit,
        }
    }

In [36]:
# -----------------------------
# 7. confirmed_trades 생성
# -----------------------------

def extract_confirmed_trades_df(result):
    model = result["model"]

    if model.status != GRB.OPTIMAL:
        print("최적해가 없습니다.")
        return pd.DataFrame()

    A_df = result["A_df"]
    q = result["q"]

    records = []
    trade_no = 1

    for row in A_df.itertuples(index=False):
        key = (row.order_id, row.seller_id, row.sku)
        qty = q[key].X

        if qty <= 1e-6:
            continue

        unit_surplus = row.unit_bid - row.ask_price
        trade_surplus = unit_surplus * qty
        weight = row.weight_per_unit * qty

        variable_delivery_cost = row.var_cost_per_weight * weight

        records.append({
            "trade_id": f"T{trade_no:04d}",
            "time_window": row.time_window,

            "order_id": row.order_id,
            "buyer_id": row.buyer_id,
            "buyer_name": row.buyer_name,
            "buyer_region": row.buyer_region,

            "seller_id": row.seller_id,
            "seller_name": row.seller_name,
            "seller_region": row.seller_region,

            "sku": row.sku,
            "item": row.item,

            "quantity": qty,
            "weight": weight,

            "unit_bid": row.unit_bid,
            "ask_price": row.ask_price,
            "unit_trade_surplus": unit_surplus,
            "trade_surplus": trade_surplus,

            "fixed_cost_base": row.fixed_cost,
            "variable_delivery_cost": variable_delivery_cost,
            "estimated_delivery_cost": variable_delivery_cost,

            "platform_unit_fee": platform_surplus_share * unit_surplus,
            "seller_unit_surplus": seller_surplus_share * unit_surplus,
            "buyer_unit_surplus": buyer_surplus_share * unit_surplus,
        })

        trade_no += 1

    df = pd.DataFrame(records)

    if df.empty:
        return df

    # buyer-seller 연결별 fixed cost를 weight 기준으로 배분
    df["fixed_delivery_cost_allocated"] = 0.0

    for (b, s), group in df.groupby(["buyer_id", "seller_id"]):
        fixed_cost = group["fixed_cost_base"].iloc[0]
        total_weight = group["weight"].sum()

        if total_weight > 1e-9:
            alloc = group["weight"] / total_weight * fixed_cost
        else:
            alloc = fixed_cost / len(group)

        df.loc[group.index, "fixed_delivery_cost_allocated"] = alloc

    df["estimated_delivery_cost"] = (
        df["variable_delivery_cost"]
        + df["fixed_delivery_cost_allocated"]
    )

    df["platform_fee_total"] = (
        df["platform_unit_fee"] * df["quantity"]
    )

    df["estimated_platform_profit_after_delivery"] = (
        df["platform_fee_total"] - df["estimated_delivery_cost"]
    )

    return df

In [37]:
# -----------------------------
# 8. 재고 업데이트
# -----------------------------

def update_inventory_df(seller_inventory_df, confirmed_trades_df):
    updated = seller_inventory_df.copy()

    if confirmed_trades_df.empty:
        return updated

    used = (
        confirmed_trades_df
        .groupby(["seller_id", "sku"])["quantity"]
        .sum()
        .reset_index()
    )

    updated = updated.merge(
        used,
        on=["seller_id", "sku"],
        how="left"
    )

    updated["quantity"] = updated["quantity"].fillna(0)
    updated["inventory"] = updated["inventory"] - updated["quantity"]
    updated = updated.drop(columns=["quantity"])

    return updated

In [38]:
# -----------------------------
# 9. 배송 bundle 생성
# -----------------------------

def build_delivery_bundles_df(confirmed_trades_df):
    if confirmed_trades_df.empty:
        return pd.DataFrame(), pd.DataFrame()

    bundle_rows = []
    item_rows = []

    group_cols = ["buyer_id", "buyer_region", "time_window"]

    for i, (key, group) in enumerate(
        confirmed_trades_df.groupby(group_cols),
        start=1
    ):
        buyer_id, buyer_region, time_window = key
        bundle_id = f"G{i:04d}"

        bundle_rows.append({
            "bundle_id": bundle_id,
            "buyer_id": buyer_id,
            "buyer_region": buyer_region,
            "time_window": time_window,
            "order_ids": sorted(group["order_id"].unique().tolist()),
            "trade_ids": sorted(group["trade_id"].unique().tolist()),
            "seller_ids": sorted(group["seller_id"].unique().tolist()),
            "seller_regions": sorted(group["seller_region"].unique().tolist()),
            "total_quantity": group["quantity"].sum(),
            "total_weight": group["weight"].sum(),
            "total_estimated_delivery_cost": group["estimated_delivery_cost"].sum(),
            "pickup_count": group["seller_id"].nunique(),
        })

        for row in group.itertuples(index=False):
            item_rows.append({
                "bundle_id": bundle_id,
                "trade_id": row.trade_id,
                "order_id": row.order_id,
                "buyer_id": row.buyer_id,
                "seller_id": row.seller_id,
                "sku": row.sku,
                "quantity": row.quantity,
                "weight": row.weight,
            })

    return pd.DataFrame(bundle_rows), pd.DataFrame(item_rows)

In [39]:
# -----------------------------
# 10. T1_AM 실행
# -----------------------------

stage1_result = solve_stage1_wdp_trading_matching_df(
    target_time_window="T1_AM",
    orders_df=orders_df,
    order_items_df=order_items_df,
    buyers_df=buyers_df,
    sellers_df=sellers_df,
    seller_inventory_df=seller_inventory_df,
    skus_df=skus_df,
    region_cost_df=region_cost_df,
    eta=1.0,
    margin_mu=0.0,
    write_lp=True
)

confirmed_trades_df = extract_confirmed_trades_df(stage1_result)

display(confirmed_trades_df)

Gurobi Optimizer version 13.0.1 build v13.0.1rc0 (win64 - Windows 11+.0 (26200.2))

CPU model: AMD Ryzen 9 5900X 12-Core Processor, instruction set [SSE2|AVX|AVX2]
Thread count: 12 physical cores, 24 logical processors, using up to 24 threads

Optimize a model with 108 rows, 53 columns and 257 nonzeros (Max)
Model fingerprint: 0xd689f974
Model has 23 linear objective coefficients
Variable types: 19 continuous, 34 integer (34 binary)
Coefficient statistics:
  Matrix range     [1e+00, 5e+05]
  Objective range  [1e+03, 9e+03]
  Bounds range     [1e+00, 1e+00]
  RHS range        [5e+01, 1e+02]

Found heuristic solution: objective -0.0000000
Presolve removed 41 rows and 16 columns
Presolve time: 0.00s
Presolved: 67 rows, 37 columns, 181 nonzeros
Variable types: 15 continuous, 22 integer (22 binary)

Root relaxation: objective 3.807831e+05, 18 iterations, 0.00 seconds (0.00 work units)

    Nodes    |    Current Node    |     Objective Bounds      |     Work
 Expl Unexpl |  Obj  Depth IntInf

,trade_id,time_window,order_id,buyer_id,buyer_name,buyer_region,seller_id,seller_name,seller_region,sku,...,trade_surplus,fixed_cost_base,variable_delivery_cost,estimated_delivery_cost,platform_unit_fee,seller_unit_surplus,buyer_unit_surplus,fixed_delivery_cost_allocated,platform_fee_total,estimated_platform_profit_after_delivery
0,T0001,T1_AM,O1,B1,수하주1,SEOUL,S3,화주3,GANGWON,P1,...,110000.000000,9000,4500.000000,13500.000000,440.0,880.0,880.0,9000.000000,22000.000000,8500.000000
1,T0002,T1_AM,O1,B1,수하주1,SEOUL,S4,화주4,GYEONGGI,P2,...,57142.857143,5000,1142.857143,6142.857143,400.0,800.0,800.0,5000.000000,11428.571429,5285.714286
2,T0003,T1_AM,O2,B2,수하주2,GYEONGGI,S3,화주3,GANGWON,P1,...,68413.793103,8500,2907.586207,11407.586207,400.0,800.0,800.0,8500.000000,13682.758621,2275.172414
3,T0004,T1_AM,O2,B2,수하주2,GYEONGGI,S2,화주2,GYEONGGI,P3,...,57600.000000,3500,840.000000,4340.000000,240.0,480.0,480.0,3500.000000,11520.000000,7180.000000
4,T0005,T1_AM,O5,B4,수하주4,SEOUL,S4,화주4,GYEONGGI,P2,...,77000.000000,5000,1400.000000,3902.102607,440.0,880.0,880.0,2502.102607,15400.000000,11497.897393
5,T0006,T1_AM,O5,B4,수하주4,SEOUL,S4,화주4,GYEONGGI,P4,...,53576.470588,5000,1397.647059,3895.544452,460.0,920.0,920.0,2497.897393,10715.294118,6819.749666


In [40]:
delivery_bundles_df, bundle_items_df = build_delivery_bundles_df(
    confirmed_trades_df
)

display(delivery_bundles_df)
display(bundle_items_df)

,bundle_id,buyer_id,buyer_region,time_window,order_ids,trade_ids,seller_ids,seller_regions,total_quantity,total_weight,total_estimated_delivery_cost,pickup_count
0,G0001,B1,SEOUL,T1_AM,[O1],"[T0001, T0002]","[S3, S4]","[GANGWON, GYEONGGI]",78.571429,72.857143,19642.857143,2
1,G0002,B2,GYEONGGI,T1_AM,[O2],"[T0003, T0004]","[S2, S3]","[GANGWON, GYEONGGI]",82.206897,58.206897,15747.586207,2
2,G0003,B4,SEOUL,T1_AM,[O5],"[T0005, T0006]",[S4],[GYEONGGI],58.294118,55.952941,7797.647059,1


,bundle_id,trade_id,order_id,buyer_id,seller_id,sku,quantity,weight
0,G0001,T0001,O1,B1,S3,P1,50.000000,50.000000
1,G0001,T0002,O1,B1,S4,P2,28.571429,22.857143
2,G0002,T0003,O2,B2,S3,P1,34.206897,34.206897
3,G0002,T0004,O2,B2,S2,P3,48.000000,24.000000
4,G0003,T0005,O5,B4,S4,P2,35.000000,28.000000
5,G0003,T0006,O5,B4,S4,P4,23.294118,27.952941


In [41]:
seller_inventory_after_T1 = update_inventory_df(
    seller_inventory_df,
    confirmed_trades_df
)

display(seller_inventory_after_T1)

,seller_id,sku,inventory,ask_price,lot_min_ratio
0,S1,P1,100.000000,4000,0.2
1,S1,P2,50.000000,5200,0.2
2,S2,P1,80.000000,4300,0.2
3,S2,P3,72.000000,3000,0.2
4,S3,P1,5.793103,3800,0.2
5,S3,P4,70.000000,6000,0.2
6,S4,P2,36.428571,5000,0.2
7,S4,P4,36.705882,6200,0.2
